# Plotly Wrapper Showcase

Demonstrates the `mayutils.visualisation.graphs.plotly` API — custom traces,
multi-axis plots, subplot grids, 3D meshes, and chart enhancements.

In [2]:
import numpy as np
import plotly.graph_objects as go

from mayutils.visualisation.graphs.plotly import (
    Bar3d,
    Cuboid,
    Ecdf,
    Icicle,
    Kde,
    Line,
    MainAxisConfig,
    MainAxisConfigs,
    Plot,
    PlotConfig,
    Scatter,
    SubPlot,
    SubPlotConfig,
    Titles,
    TracesConfig,
    merge_cuboids,
)

rng = np.random.default_rng(seed=343)

## 1. Plot.from_traces — quick single-axis chart

In [3]:
Plot.from_traces(
    *(
        Scatter(
            x=(x := rng.normal(loc=50 + i * 15, scale=12, size=80)),
            y=x + rng.normal(scale=8, size=80),
            name=name,
            marker={"opacity": 0.7, "line_width": 0},
        )
        for i, name in enumerate(["Segment A", "Segment B", "Segment C"])
    ),
    description="scatter-basics",
    xaxis_config={"title_text": "Feature X"},
    yaxis_config={"title_text": "Feature Y"},
    layout={"title_text": "Scatter Plot"},
)

## 2. Line with confidence bounds

In [4]:
x = np.linspace(0, 10, 60)
y = np.sin(x) * 10 + 50

Plot.from_traces(
    *Line.with_bounds(
        x=x,
        y=y,
        y_upper=(y + 3, y + 7),
        y_lower=(y - 3, y - 7),
        name="Forecast",
    ),
    description="line-bounds",
    xaxis_config={"title_text": "Time"},
    yaxis_config={"title_text": "Value"},
    layout={"title_text": "Line with Confidence Bounds"},
)

## 3. Multi-y-axis plot

In [ ]:
t = np.linspace(0, 4 * np.pi, 120)

Plot(
    PlotConfig(
        yaxes_configs=(
            TracesConfig(
                traces=(Line(x=t, y=np.sin(t) * 100, name="Revenue"),),
                yaxis_config={"title_text": "Revenue ($)"},
            ),
            TracesConfig(
                traces=(Line(x=t, y=np.cos(t) * 50 + 50, name="Margin (%)"),),
                yaxis_config={"title_text": "Margin (%)"},
            ),
            TracesConfig(
                traces=(Line(x=t, y=np.cumsum(rng.normal(size=120)), name="Volume"),),
                yaxis_config={"title_text": "Volume"},
            ),
        ),
        xaxis_config={"title_text": "Period"},
    ),
    description="multi-y-axis",
    layout={"title_text": "Three Y-Axes"},
)

## 4. Histogram with gaussians, KDE, and rug

In [ ]:
(
    Plot.from_traces(
        go.Histogram(x=rng.normal(loc=0, scale=1, size=500), nbinsx=80, bingroup=1, name="Main"),  # pyright: ignore[reportArgumentType]
        go.Histogram(x=rng.normal(loc=3, scale=0.6, size=200), nbinsx=80, bingroup=1, name="Secondary"),  # pyright: ignore[reportArgumentType]
        go.Histogram(x=rng.normal(loc=-2, scale=0.4, size=100), nbinsx=80, bingroup=1, name="Tertiary"),  # pyright: ignore[reportArgumentType]
        description="histogram-extras",
        layout={"title_text": "Histogram + Gaussian Fits + KDE + Rug"},
    )
    .add_rug(rug_type="violin")
    .add_default_extras()
    .add_histogram_gaussians()
)

## 5. ECDF and KDE traces

In [ ]:
samples = {
    "Normal": rng.normal(loc=5, scale=1.5, size=300),
    "Skewed": rng.exponential(scale=2, size=300),
    "Bimodal": np.concatenate([rng.normal(3, 0.8, 150), rng.normal(8, 1.0, 150)]),
}

Plot(
    PlotConfig(
        yaxes_configs=(
            TracesConfig(
                traces=tuple(Ecdf(x=v, name=k) for k, v in samples.items()),
                yaxis_config={"title_text": "Cumulative Probability"},
            ),
            TracesConfig(
                traces=tuple(Kde(x=v, name=f"{k} (KDE)") for k, v in samples.items()),
                yaxis_config={"title_text": "Density"},
            ),
        ),
        xaxis_config={"title_text": "Value"},
    ),
    description="ecdf-kde",
    layout={"title_text": "ECDF + KDE Distributions"},
)

## 6. Icicle chart from nested dict

In [ ]:
Plot.from_traces(
    Icicle.from_dict(  # pyright: ignore[reportArgumentType]
        icicle_dict={
            "UK": {
                "England": {"London": 9.0, "Manchester": 2.7, "Birmingham": 1.1},
                "Scotland": {"Edinburgh": 0.5, "Glasgow": 0.6},
                "Wales": {"Cardiff": 0.4},
            },
            "Germany": {
                "Bavaria": {"Munich": 1.5},
                "Berlin": {"Berlin": 3.6},
            },
        },
        name="Population (M)",
    ),
    description="icicle",
    layout={"title_text": "Icicle — Nested Hierarchy"},
)

## 7. 3D Bar chart and merged Cuboids

In [ ]:
x_cat = np.array(["Mon", "Tue", "Wed", "Thu", "Fri"])
y_cat = np.array(["Q1", "Q2", "Q3", "Q4"])
xg, yg = np.meshgrid(x_cat, y_cat)
z_vals = rng.integers(1, 10, size=xg.size).astype(float)

Plot.from_traces(
    Bar3d(  # pyright: ignore[reportArgumentType]
        x=xg.ravel(),
        y=yg.ravel(),
        z=z_vals,
        name="3D Bars",
    ),
    description="bar3d",
    layout={"title_text": "3D Bar Chart"},
)

In [ ]:
cuboids = [
    Cuboid(x=(0, 2), y=(0, 3), z=(0, 5), weight=0.8, name="A"),
    Cuboid(x=(3, 5), y=(0, 3), z=(0, 3), weight=0.4, name="B"),
    Cuboid(x=(0, 2), y=(4, 6), z=(0, 7), weight=1.0, name="C"),
]

Plot.from_traces(
    merge_cuboids(*cuboids),  # pyright: ignore[reportArgumentType]
    description="merged-cuboids",
    layout={"title_text": "Merged Cuboids"},
)

## 8. Subplot grid with row/column/cell titles

In [11]:
SubPlot(
    SubPlotConfig(
        plots=(
            (
                PlotConfig.from_traces(
                    Line(x=t, y=np.sin(t), name="sin"),
                    Line(x=t, y=np.cos(t), name="cos"),
                ),
                PlotConfig.from_trace(
                    Scatter(x=rng.normal(size=60), y=rng.normal(size=60), name="noise"),
                ),
                PlotConfig.from_trace(
                    Line(x=t, y=np.cumsum(rng.normal(size=len(t))), name="walk"),
                ),
            ),
            (
                PlotConfig.from_trace(
                    Line(x=t, y=np.sin(t) ** 2, name="sin^2"),
                ),
                PlotConfig.from_trace(
                    Scatter(
                        x=rng.uniform(size=60),
                        y=rng.exponential(size=60),
                        name="exp",
                    ),
                ),
                PlotConfig.from_trace(
                    Line(x=t, y=np.tanh(t / 4), name="tanh"),
                ),
            ),
        ),
        main_axis_configs=MainAxisConfigs(
            xaxis=MainAxisConfig.from_dict(mode="independent"),
            yaxes=(MainAxisConfig.from_dict(mode="independent"),),
        ),
        titles=Titles(
            main="Subplot Grid",
            rows=("Row A", "Row B"),
            cols=("Periodic", "Random", "Cumulative"),
        ),
    ),
    description="subplot-grid",
    layout={"height": 600, "width": 1000},
)

## 9. SubPlotConfig.flat — auto-grid layout

In [12]:
cells = tuple(
    PlotConfig.from_trace(
        Line(
            x=np.linspace(0, 10, 50),
            y=np.sin(np.linspace(0, 10, 50) + phase),
            name=f"phase={phase:.1f}",
        ),
    )
    for phase in np.linspace(0, 2 * np.pi, 6)
)

SubPlot(
    SubPlotConfig.flat(
        cells,
        cols=3,
        titles=Titles(main="Auto-Grid (SubPlotConfig.flat)"),
    ),
    description="flat-grid",
    layout={"height": 500, "width": 900},
)

## 10. Subplot with 3D surface + 2D panels

In [13]:
xs = np.linspace(-3, 3, 40)
ys = np.linspace(-3, 3, 40)
xg, yg = np.meshgrid(xs, ys)
zg = np.sin(xg) * np.cos(yg)

SubPlot(
    SubPlotConfig(
        plots=(
            (
                PlotConfig.from_trace(go.Surface(x=xs, y=ys, z=zg, showscale=False)),
                PlotConfig.from_traces(
                    Scatter(x=rng.normal(size=100), y=rng.normal(size=100), name="cluster A"),
                    Scatter(x=rng.normal(2, 0.5, 100), y=rng.normal(2, 0.5, 100), name="cluster B"),
                ),
            ),
        ),
        titles=Titles(main="Mixed 3D + 2D Subplot"),
    ),
    description="mixed-3d-2d",
    layout={"height": 450, "width": 900},
)

## 11. Dropdown toggle (Plot.as_dropdown)

In [14]:
linear = Plot.from_traces(
    Line(x=t, y=t, name="y = x"),
    description="linear",
    layout={"title_text": "Linear"},
)
quadratic = Plot.from_traces(
    Line(x=t, y=t**2, name="y = x^2"),
    description="quadratic",
    layout={"title_text": "Quadratic"},
)
sinusoidal = Plot.from_traces(
    Line(x=t, y=np.sin(t) * 5, name="y = sin(x)"),
    description="sinusoidal",
    layout={"title_text": "Sinusoidal"},
)

Plot.as_dropdown(
    "dropdown-demo",
    Linear=linear,
    Quadratic=quadratic,
    Sinusoidal=sinusoidal,
)

## 12. Scatter with marginal histograms

In [15]:
Plot.from_traces(
    Scatter(
        x=rng.normal(loc=5, scale=2, size=200),
        y=rng.normal(loc=10, scale=3, size=200),
        name="observations",
        marker={"opacity": 0.5},
    ),
    description="scatter-hist",
    layout={"title_text": "Scatter + Marginal Histograms"},
).add_histogram_to_2d_scatter()